# 03 - Combined Pipeline (current shot + next shot)

Runs on **Google Colab with a T4 GPU**. Loads the two trained models from Drive, classifies a clip, keeps a 4-slot rally history, predicts the next shot, and runs a combined evaluation over the labelled clip folders.

Run notebooks 01 and 02 first so the checkpoints exist.

In [7]:
import torch
assert torch.cuda.is_available(), 'No GPU active. In Colab: Runtime > Change runtime type > T4 GPU, then re-run this cell.'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless

GPU: NVIDIA L4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
CLASSES = [
    (0, 'Short Serve', '00_Short Serve'),
    (1, 'Lift',        '02_Lift'),
    (2, 'Clear',       '12_Clear'),
    (3, 'Drop Shot',   '05_Drop Shot'),
    (4, 'Smash',       '14_Smash'),
]
NUM_CLASSES   = len(CLASSES)
DISPLAY_NAMES = [c[1] for c in CLASSES]
CLASS_FOLDERS = [c[2] for c in CLASSES]
ID_TO_NAME    = {c[0]: c[1] for c in CLASSES}
FOLDER_TO_ID  = {c[2]: c[0] for c in CLASSES}
PAD_TOKEN     = NUM_CLASSES
SEQ_LEN       = 4
NUM_FRAMES    = 16
IMG           = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DRIVE       = '/content/drive/MyDrive/BadmintonFYP'
MODELS      = DRIVE + '/model23'
VIDEO_CKPT    = DRIVE + '/model23/video_model.pth'
SEQUENCE_CKPT = MODELS + '/sequence_model.pth'

import os, glob
def _has(d): return os.path.isdir(d) and any(os.path.isdir(os.path.join(d, f)) for f in CLASS_FOLDERS)
def _find(base):
    for c in [base] + sorted(glob.glob(base + '/*')) + sorted(glob.glob(base + '/*/*')):
        if _has(c): return c
    return None
DATASET_DIR = _find('/content/Dataset') or _find('/content') or _find(DRIVE) or '/content/Dataset'
print('Models dir:', MODELS)
print('DATASET_DIR:', DATASET_DIR)

Models dir: /content/drive/MyDrive/BadmintonFYP/model23
DATASET_DIR: /content/data


In [9]:
import os, glob
from collections import deque, Counter
import cv2, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TemporalPoolingResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        base = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        x = self.features(x).view(b, t, 512).mean(dim=1)
        return self.fc(x)

class ShotPredictorGRU(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed=32, hidden=128, layers=2, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(num_classes + 1, embed, padding_idx=PAD_TOKEN)
        self.gru = nn.GRU(embed, hidden, layers, batch_first=True, dropout=drop if layers > 1 else 0)
        self.fc = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(drop), nn.Linear(hidden, num_classes))
    def forward(self, x):
        x = self.emb(x); o, _ = self.gru(x); return self.fc(o[:, -1, :])

_norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
def load_video_frames(path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): raise ValueError('cannot open ' + path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0: cap.release(); raise ValueError('0 frames ' + path)
    idxs = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap.read()
        if ok: frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        elif frames: frames.append(frames[-1].copy())
    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1].copy() if frames else np.zeros((IMG, IMG, 3), np.uint8))
    return frames[:num_frames]
def letterbox(img, size=IMG):
    h, w = img.shape[:2]; s = min(size / h, size / w); nh, nw = int(h * s), int(w * s)
    r = cv2.resize(img, (nw, nh)); canvas = np.zeros((size, size, 3), np.uint8)
    y, x = (size - nh) // 2, (size - nw) // 2; canvas[y:y+nh, x:x+nw] = r; return canvas
def preprocess_clip(path):
    out = []
    for fr in load_video_frames(path):
        fr = letterbox(fr).astype(np.float32) / 255.0
        out.append(_norm(torch.from_numpy(fr).permute(2, 0, 1)))
    return torch.stack(out).unsqueeze(0)
print('defs ready')

defs ready


In [ ]:
def load_state(model, path):
    ck = torch.load(path, map_location=DEVICE)
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    model.load_state_dict(state); model.eval(); return model

video_model = load_state(TemporalPoolingResNet().to(DEVICE), VIDEO_CKPT)
print('loaded video model')
seq_model = None
if os.path.exists(SEQUENCE_CKPT):
    seq_model = load_state(ShotPredictorGRU().to(DEVICE), SEQUENCE_CKPT)
    print('loaded sequence model')
else:
    print('sequence model missing - run notebook 02 (next-shot disabled)')

def topk(probs, k=3):
    k = min(k, probs.numel()); v, idx = probs.topk(k)
    return [{'id': int(i), 'label': ID_TO_NAME[int(i)], 'prob': float(p)} for p, i in zip(v, idx)]

def predict_current(path, k=3):
    x = preprocess_clip(path).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(video_model(x), dim=1)[0].cpu()
    t = topk(probs, k)
    return {'top': t, 'pred_id': t[0]['id'], 'label': t[0]['label'], 'confidence': t[0]['prob']}

def predict_next(history_ids, k=3):
    h = list(history_ids)[-SEQ_LEN:]
    if len(h) < SEQ_LEN: h = [PAD_TOKEN] * (SEQ_LEN - len(h)) + h
    x = torch.tensor([h], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        probs = F.softmax(seq_model(x), dim=1)[0].cpu()
    return topk(probs, k)

class RallyHistory:
    def __init__(self, maxlen=SEQ_LEN): self.buf = deque(maxlen=maxlen)
    def append(self, i): self.buf.append(int(i))
    def ids(self): return list(self.buf)
    def labels(self): return [ID_TO_NAME[i] for i in self.buf]
    def reset(self): self.buf.clear()

def coaching_line(cur, nxt):
    r = {'Smash': 'Attack played - opponent likely forced into ' + nxt,
         'Drop Shot': 'Net pressure applied - expect ' + nxt,
         'Short Serve': 'Rally started - anticipate ' + nxt,
         'Lift': 'Lifted to the back - prepare for ' + nxt,
         'Clear': 'Cleared deep - next likely ' + nxt}
    return r.get(cur, 'Next likely shot: ' + nxt)


def analyze_clip(path, history):
    cur = predict_current(path); history.append(cur['pred_id'])
    nxt = coach = None
    if seq_model is not None:
        nxt = predict_next(history.ids()); coach = coaching_line(cur['label'], nxt[0]['label'])
    return {'clip': os.path.basename(path), 'current': cur, 'history': history.labels(), 'next': nxt, 'coaching': coach}
print('pipeline ready')

loaded video model
loaded sequence model
pipeline ready


In [11]:
hist = RallyHistory()
sample = sorted(glob.glob(DATASET_DIR + '/*/*.mp4'))
if sample:
    r = analyze_clip(sample[0], hist)
    print('clip      :', r['clip'])
    print('current   :', r['current']['label'], '(%.1f%%)' % (100 * r['current']['confidence']))
    print('top-3     :', [(p['label'], round(p['prob'], 3)) for p in r['current']['top']])
    print('history   :', r['history'])
    if r['next']: print('next      :', [(p['label'], round(p['prob'], 3)) for p in r['next']])
    if r['coaching']: print('coaching  :', r['coaching'])
else:
    print('No clips under', DATASET_DIR, '- run notebook 01 first (it unzips the dataset).')

clip      : 2022-08-30_18-00-09_dataset_set1_009_001424_001452_A_00.mp4
current   : Short Serve (99.9%)
top-3     : [('Short Serve', 0.999), ('Smash', 0.0), ('Lift', 0.0)]
history   : ['Short Serve']
next      : [('Drop Shot', 0.338), ('Lift', 0.325), ('Smash', 0.223)]
coaching  : Rally started - anticipate Drop Shot


In [12]:
PER_CLASS = 5
results, correct = [], 0
for folder in CLASS_FOLDERS:
    true_id = FOLDER_TO_ID[folder]
    for clip in sorted(glob.glob(os.path.join(DATASET_DIR, folder, '*.mp4')))[:PER_CLASS]:
        r = analyze_clip(clip, RallyHistory())
        ok = r['current']['pred_id'] == true_id; correct += ok
        results.append((ID_TO_NAME[true_id], r['current']['label'], round(r['current']['confidence'], 3), ok))
n = max(len(results), 1)
print('Combined current-shot accuracy: %d/%d (%.1f%%)' % (correct, len(results), 100 * correct / n))
print()
print('%-13s%-13s%-7s%s' % ('True', 'Pred', 'Conf', 'OK'))
for t, p, c, ok in results:
    print('%-13s%-13s%-7s%s' % (t, p, c, 'Y' if ok else 'N'))
print()
print('prediction distribution:', dict(Counter(p for _, p, _, _ in results)))

Combined current-shot accuracy: 25/25 (100.0%)

True         Pred         Conf   OK
Short Serve  Short Serve  0.999  Y
Short Serve  Short Serve  1.0    Y
Short Serve  Short Serve  0.997  Y
Short Serve  Short Serve  0.995  Y
Short Serve  Short Serve  0.999  Y
Lift         Lift         0.998  Y
Lift         Lift         0.993  Y
Lift         Lift         1.0    Y
Lift         Lift         1.0    Y
Lift         Lift         0.974  Y
Clear        Clear        0.999  Y
Clear        Clear        0.999  Y
Clear        Clear        1.0    Y
Clear        Clear        0.999  Y
Clear        Clear        0.996  Y
Drop Shot    Drop Shot    0.995  Y
Drop Shot    Drop Shot    0.959  Y
Drop Shot    Drop Shot    0.921  Y
Drop Shot    Drop Shot    0.977  Y
Drop Shot    Drop Shot    0.996  Y
Smash        Smash        1.0    Y
Smash        Smash        1.0    Y
Smash        Smash        1.0    Y
Smash        Smash        0.977  Y
Smash        Smash        0.963  Y

prediction distribution: {'Short Serve':